In [5]:
import json

# The "Control Panel" for the Generator
# In production, you would save this as 'config.json'
PIPELINE_CONFIG = {
    "seed": 42,
    "n_sites": 5,
    "n_employees": 1000,
    "start_date": "2024-01-01",
    "end_date": "2025-12-31",
    "coefficients": {
        "beta_0": -10.5,
        "beta_tenure": -0.02,
        "beta_pressure": 2.0,
        "gamma_0": 0.5,
        "gamma_severity": 2.0,
        "gamma_pressure": -1.5
    }
}

In [6]:
import pandas as pd
import numpy as np
from scipy.special import expit
import logging
from typing import Dict, Any

# Set up logging to track the generator's progress
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class WorkplaceSafetyGenerator:
    """
    Generates synthetic Environment, Health, and Safety (EHS) datasets 
    using structural equation modeling to simulate reporting bias.
    """
    
    def __init__(self, config: Dict[str, Any]):
        """Initializes the generator with a configuration dictionary."""
        self.config = config
        self.coef = config['coefficients']
        np.random.seed(self.config['seed'])
        
        # Internal state (DataFrames)
        self.sites_df = None
        self.emps_df = None
        self.raw_incidents = None
        self.final_incidents = None
        self.investigations_df = None
        self.actions_df = None
        
        logging.info(f"Initialized Generator with seed {self.config['seed']}")

    def generate_base_tables(self) -> None:
        """Generates the Sites and Employees tables."""
        logging.info(f"Generating {self.config['n_sites']} Sites and {self.config['n_employees']} Employees...")
        
        # 1. Sites Table
        self.sites_df = pd.DataFrame({
            'site_id': np.arange(1, self.config['n_sites'] + 1),
            'region': np.random.choice(['Luzon', 'Visayas', 'Mindanao'], size=self.config['n_sites'], p=[0.5, 0.3, 0.2]),
            'facility_type': np.random.choice(['Heavy Assembly', 'Chemical Processing', 'Warehousing'], size=self.config['n_sites']),
            'site_age_years': np.random.randint(5, 40, size=self.config['n_sites']),
            'site_culture_latent': np.random.uniform(0.1, 0.5, size=self.config['n_sites']) # Omitted later
        })
        
        # 2. Employees Table
        self.emps_df = pd.DataFrame({
            'emp_id': np.arange(1, self.config['n_employees'] + 1),
            'site_id': np.random.choice(self.sites_df['site_id'], size=self.config['n_employees']),
            'tenure_months': np.random.randint(1, 120, size=self.config['n_employees']),
            'job_role': np.random.choice(['Operator', 'Technician', 'Warehouse', 'Supervisor'], size=self.config['n_employees']),
            'shift_type': np.random.choice(['Day', 'Night', 'Swing'], size=self.config['n_employees'], p=[0.6, 0.25, 0.15]),
            'last_training_score': np.clip(np.random.normal(85, 10, size=self.config['n_employees']), 0, 100).round(1)
        })

    def generate_true_incidents(self) -> None:
        """Calculates latent pressure and true accident rates via Poisson point process."""
        logging.info("Calculating latent pressure and simulating 'True' incidents...")
        
        dates = pd.date_range(self.config['start_date'], self.config['end_date'])
        
        # Create Person-Day Grid
        grid = pd.MultiIndex.from_product(
            [self.emps_df['emp_id'], dates], names=['emp_id', 'date']
        ).to_frame().reset_index(drop=True)
        
        df = grid.merge(self.emps_df, on='emp_id').merge(self.sites_df, on='site_id')
        
        # Calculate Latent Pressure (Quota Spike logic)
        df['day_of_month'] = df['date'].dt.day
        df['latent_pressure'] = (df['site_culture_latent'] + 
                                (0.05 * np.exp(df['day_of_month'] / 8)) + 
                                np.random.normal(0, 0.05, len(df))).clip(0, 1)
        
        # Calculate Incident Rate (Lambda)
        log_lambda = (self.coef['beta_0'] + 
                      self.coef['beta_tenure'] * (df['tenure_months']/12) + 
                      self.coef['beta_pressure'] * df['latent_pressure'])
        
        df['has_incident'] = np.random.random(len(df)) < np.exp(log_lambda)
        
        # Keep only events where an incident occurred
        self.raw_incidents = df[df['has_incident'] == True].copy()

    def apply_reporting_bias(self) -> None:
        """Applies the reporting filter, assigns severity, and drops omitted variables."""
        logging.info("Applying reporting bias and hiding omitted variables...")
        df = self.raw_incidents.copy()
        n_obs = len(df)
        
        # Assign categorical 'Paint'
        df['incident_type'] = np.random.choice(['Fall from Height', 'Caught-in-Machine', 'Chemical Exposure', 'Laceration', 'Slips/Trips'], size=n_obs, p=[0.1, 0.15, 0.05, 0.4, 0.3])
        df['body_part'] = np.random.choice(['Hands/Fingers', 'Eyes', 'Back', 'Lower Limbs', 'Head'], size=n_obs, p=[0.4, 0.1, 0.2, 0.2, 0.1])
        df['severity_level'] = np.random.choice([1, 2, 3, 4], size=n_obs, p=[0.7, 0.2, 0.09, 0.01])
        
        # Latent Pressure influences PPE Compliance
        ppe_logits = 2.0 - 1.5 * df['latent_pressure']
        df['ppe_compliant'] = np.random.random(n_obs) < expit(ppe_logits)
        
        # Reporting Logic
        reporting_logits = (self.coef['gamma_0'] + 
                           self.coef['gamma_severity'] * df['severity_level'] + 
                           self.coef['gamma_pressure'] * df['latent_pressure'])
        df['is_reported'] = np.random.random(n_obs) < expit(reporting_logits)
        
        # Filter observed data and assign Primary Key
        observed = df[df['is_reported'] == True].copy()
        observed['inc_id'] = np.arange(1, len(observed) + 1)
        
        # Drop Omitted Variables
        cols_to_remove = ['site_culture_latent', 'latent_pressure', 'is_reported', 'has_incident', 'day_of_month']
        self.final_incidents = observed.drop(columns=cols_to_remove, errors='ignore')
        
        # Clean Sites table as well
        self.sites_df = self.sites_df.drop(columns=['site_culture_latent'], errors='ignore')

    def generate_post_incident_data(self) -> None:
        """Generates Investigations and Corrective Actions for reported incidents."""
        logging.info("Generating Investigations and Corrective Actions...")
        n_inv = len(self.final_incidents)
        
        # Investigations
        self.investigations_df = pd.DataFrame({
            'inv_id': np.arange(1, n_inv + 1),
            'inc_id': self.final_incidents['inc_id'].values,
            'primary_cause': np.random.choice(['Equipment Failure', 'Lack of Training', 'Bypassing Guardrails', 'Fatigue', 'Poor Housekeeping'], size=n_inv, p=[0.2, 0.25, 0.2, 0.1, 0.25]),
            'investigator_rating': np.random.randint(1, 6, size=n_inv),
            'days_to_complete': np.random.lognormal(mean=2.8, sigma=0.8, size=n_inv).astype(int) + 1
        })
        
        # Actions (1-to-Many Vectorized)
        actions_per_inv = np.random.choice([1, 2, 3], size=n_inv, p=[0.5, 0.3, 0.2])
        inv_ids_repeated = np.repeat(self.investigations_df['inv_id'].values, actions_per_inv)
        n_act = len(inv_ids_repeated)
        
        self.actions_df = pd.DataFrame({
            'action_id': np.arange(1, n_act + 1),
            'inv_id': inv_ids_repeated,
            'action_category': np.random.choice(['Engineering Control', 'Administrative', 'Training', 'PPE Issuance'], size=n_act),
            'action_cost_usd': np.round(np.random.lognormal(mean=7.0, sigma=1.5, size=n_act), 2),
            'completion_status': np.random.choice(['Closed', 'In Progress', 'Open'], size=n_act, p=[0.7, 0.2, 0.1])
        })

    def export_data(self) -> None:
        """Saves the 5 cleaned dataframes to CSV."""
        logging.info("Exporting to CSV...")
        self.sites_df.to_csv('sites.csv', index=False)
        self.emps_df.to_csv('employees.csv', index=False)
        self.final_incidents.to_csv('incidents.csv', index=False)
        self.investigations_df.to_csv('investigations.csv', index=False)
        self.actions_df.to_csv('corrective_actions.csv', index=False)
        logging.info("Export Complete! Data is ready for analysis.")

    def run_pipeline(self) -> None:
        """Orchestrates the entire generation process."""
        self.generate_base_tables()
        self.generate_true_incidents()
        self.apply_reporting_bias()
        self.generate_post_incident_data()
        self.export_data()

# ==========================================
# 3. HOW TO RUN THE GENERATOR
# ==========================================
if __name__ == "__main__":
    # Instantiate the class with the configuration
    generator = WorkplaceSafetyGenerator(PIPELINE_CONFIG)
    
    # Run the entire pipeline with one command
    generator.run_pipeline()

2026-04-06 13:53:07,374 - INFO - Initialized Generator with seed 42
2026-04-06 13:53:07,375 - INFO - Generating 5 Sites and 1000 Employees...
2026-04-06 13:53:07,377 - INFO - Calculating latent pressure and simulating 'True' incidents...
2026-04-06 13:53:07,499 - INFO - Applying reporting bias and hiding omitted variables...
2026-04-06 13:53:07,503 - INFO - Generating Investigations and Corrective Actions...
2026-04-06 13:53:07,504 - INFO - Exporting to CSV...
2026-04-06 13:53:07,509 - INFO - Export Complete! Data is ready for analysis.
